In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
COLLOCATE_DIR = ROOT / "data" / "gulf_stream_20240305_20260531" / "silver" / "collocate_pace"

In [ ]:
rows = []
for polarity in ("cyclone", "anticyclone"):
    for fp in sorted((COLLOCATE_DIR / polarity).glob("eddy_*_rrs.parquet")):
        df = pd.read_parquet(fp, columns=["track_id", "date"])
        tid = df["track_id"].iloc[0]
        n_dates = df["date"].nunique()
        rows.append({"track_id": tid, "polarity": polarity, "n_dates": n_dates})

eddy_dates = pd.DataFrame(rows)
print(
    f"eddies: {len(eddy_dates)}\n"
    f"total_eddy_date_instances: {eddy_dates['n_dates'].sum()}"
)
eddy_dates.groupby("polarity")["n_dates"].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (pol, color) in zip(axes, [("cyclone", "tab:blue"), ("anticyclone", "tab:red")]):
    sub = eddy_dates[eddy_dates["polarity"] == pol].sort_values("track_id")
    ax.bar(sub["track_id"].values, sub["n_dates"].values, color=color)
    ax.axhline(sub["n_dates"].median(), color="gray", ls="--", lw=1,
        label=f"median = {sub['n_dates'].median():.0f}")
    ax.set_xlabel("Track ID")
    ax.set_title(pol.capitalize())
    ax.legend()

axes[0].set_ylabel("Eddy-date instances")
fig.suptitle("PACE observations per eddy after collocation", y=1.02)
fig.tight_layout()